In [16]:
import pandas as pd
import json
import os

from lib.git_history import iterate_file_versions

In [19]:
input_file = os.path.join("data", "ch.bfe.ladestellen-elektromobilitaet-status.json")
output = os.path.join("data_cache")

status_recs = []
for i, git_commit_at, git_hash, content in iterate_file_versions(input_file):
    versioned_json = json.loads(content)
    timestamp = git_commit_at.isoformat()

    for operator in versioned_json.get("EVSEStatuses", []):
        operator_id = operator.get("OperatorID", "")
        operator_name = operator.get("OperatorName", "")

        for evse in operator.get("EVSEStatusRecord", []):
            evse_id = evse.get("EvseID", "")
            evse_status = evse.get("EVSEStatus", "")

            status_rec = {
                "Timestamp": timestamp,
                "OperatorID": operator_id,
                "OperatorName": operator_name,
                "EvseID": evse_id,
                "EVSEStatus": evse_status
            }
            status_recs.append(status_rec)

    # save every 10 versions as a pickle file
    if (i % 10) == 0:
        df_rec = df = pd.DataFrame(status_recs)
        df_rec["Timestamp"] = pd.to_datetime(df_rec["Timestamp"], utc=True)
        pickle_path = os.path.join("data_cache", f"{i}.pks")
        df_rec.to_pickle(pickle_path)
        print(f"Saved pickle at {pickle_path}")
        status_recs = []

Saved pickle at data_cache\0.pks
Saved pickle at data_cache\10.pks
Saved pickle at data_cache\20.pks
Saved pickle at data_cache\30.pks
Saved pickle at data_cache\40.pks
Saved pickle at data_cache\50.pks
Saved pickle at data_cache\60.pks
Saved pickle at data_cache\70.pks
Saved pickle at data_cache\80.pks
Saved pickle at data_cache\90.pks
Saved pickle at data_cache\100.pks
Saved pickle at data_cache\110.pks
Saved pickle at data_cache\120.pks
Saved pickle at data_cache\130.pks
Saved pickle at data_cache\140.pks
Saved pickle at data_cache\150.pks
Saved pickle at data_cache\160.pks
Saved pickle at data_cache\170.pks
Saved pickle at data_cache\180.pks
Saved pickle at data_cache\190.pks
Saved pickle at data_cache\200.pks
Saved pickle at data_cache\210.pks
Saved pickle at data_cache\220.pks
Saved pickle at data_cache\230.pks
Saved pickle at data_cache\240.pks
Saved pickle at data_cache\250.pks
Saved pickle at data_cache\260.pks
Saved pickle at data_cache\270.pks
Saved pickle at data_cache\280.

In [20]:
# Combine pickle files to one dataframe
df_records = pd.concat(
    [pd.read_pickle(os.path.join("data_cache", x)) for x in os.listdir("data_cache") if x.endswith(".pks")]
)

df_records

,Timestamp,OperatorID,OperatorName,EvseID,EVSEStatus
0,2025-10-23 09:22:48+00:00,CH*SWISSCHARGE,Swisscharge,CH*SWI*E10382,OutOfService
1,2025-10-23 09:22:48+00:00,CH*SWISSCHARGE,Swisscharge,CH*SWI*E10383,OutOfService
2,2025-10-23 09:22:48+00:00,CH*SWISSCHARGE,Swisscharge,CH*SWI*E262873,OutOfService
3,2025-10-23 09:22:48+00:00,CH*SWISSCHARGE,Swisscharge,CH*SWI*E262874,OutOfService
4,2025-10-23 09:22:48+00:00,CH*SWISSCHARGE,Swisscharge,CH*SWI*E262875,OutOfService
...,...,...,...,...,...
166665,2025-10-24 07:16:46+00:00,CH*LDL,Lidl Schweiz AG,CH*LDL*E00000014,Available
166666,2025-10-24 07:16:46+00:00,CH*LDL,Lidl Schweiz AG,CH*LDL*E00000448,Available
166667,2025-10-24 07:16:46+00:00,CH*LDL,Lidl Schweiz AG,CH*LDL*E00000130,Available
166668,2025-10-24 07:16:46+00:00,CH*LDL,Lidl Schweiz AG,CH*LDL*E00000131,Available
